# Single agent versus multi-agent systems

## Northstar incident: should checkout investigation become a team?

Checkout conversion falls 35% in Europe after a release. One well-designed investigator can use metrics, logs, deployment history, customer complaints, and runbooks. A specialist team may isolate those domains and add independent review. This notebook tests whether that complexity earns its place.

**Outcomes:** identify real reasons to add agents; design and compare centralized, decentralized, sequential, parallel, blackboard, debate, and hierarchy patterns; build scoped artifacts; and make an evidence-based promotion decision. **Safety:** all cells are deterministic; no external system is called.

![Supervisor team and evidence board](assets/team-topologies.svg)

This static SVG renders without a browser-side Mermaid extension. It depicts one useful topology, not the default answer. The README contains editable Mermaid diagrams and a full pattern catalogue.

## 1. Begin with a strong single-agent baseline

Multiple agents introduce routing, duplicated context, message/turn cost, shared-state poisoning, termination, identity, and authorization problems. First improve a single agent with a deterministic workflow, narrow tools, context routing, a critic pass, and clear stopping conditions. Add a team only for a named bottleneck: specialization, context isolation, genuine parallel work, modular ownership, or independent review.

In [1]:
from pathlib import Path
import sys
TOPIC = Path.cwd()
if not (TOPIC / 'lab.py').exists():
    TOPIC = Path.cwd() / 'curriculum' / 'advanced' / '01-single-vs-multi-agent'
sys.path.insert(0, str(TOPIC))
from lab import Artifact, Board, PATTERNS, allocate, choose, compare, critic

for complexity in ('simple', 'cross-domain', 'ambiguous'):
    print(complexity, compare(complexity)['recommended'])
assert compare('simple')['recommended'] == 'single investigator'

simple single investigator
cross-domain supervisor team
ambiguous single investigator


## 2. Why agents can help—and why they can hurt

**Specialization:** give a role a distinct tool, rubric, or domain context—not merely a new name. **Context isolation:** place private/noisy/regulated material behind a typed output contract. **Parallelization:** fan out only independent reads, then budget the join. **Modularity:** create components that can be independently owned, tested, and replaced. **Organizational modeling:** map accountable review roles, but preserve application-enforced identity, scope, approval, and audit.

A team is a distributed system. Its quality depends on contracts and coordination more than on dramatic role prompts.

## 3. Pattern selection

- **Supervisor → workers:** centralized delegation and audit for bounded specialized work.
- **Router → specialists:** classify known verticals; evaluate misroutes and fan-out.
- **Planner → executors:** validate a DAG, dependencies, milestones, and replanning budget.
- **Manager → subagents / hierarchy:** reduce context at scale; cap delegation depth and retain evidence.
- **Peer-to-peer:** local, resilient negotiation only with authenticated messages, quorum, TTL, and termination.
- **Blackboard:** shared, attributable, versioned evidence; not an unrestricted transcript.
- **Debate / generator-critic / voting:** useful only with diverse evidence, rubrics, bounded turns, and a verifier.
- **Sequential pipeline:** predictable validated handoffs, but serial latency.
- **Parallel swarm:** bounded fan-out/fan-in; requires quotas, cancellation, partial-result policy, and aggregation.

Centralized control normally makes audit and policy easier. Decentralization is a trade-off for locality or resilience, not an automatic upgrade.

In [2]:
for name, guidance in PATTERNS.items():
    print(f'{name:20} {guidance}')

assert 'blackboard' in PATTERNS and 'peer-to-peer' in PATTERNS

supervisor-workers   Central owner delegates bounded specialist work; audit-friendly but can bottleneck.
router-specialists   Classify and dispatch clear domains; measure misroutes and constrain fan-out.
planner-executors    Validated DAG plan with constrained executors; version plans and bound replans.
manager-subagents    Layered ownership reduces context; cap delegation depth and retain source artifacts.
hierarchical         Nested teams for decomposable programs; avoid summary loss and authority expansion.
peer-to-peer         Authenticated peer negotiation; needs TTL, quorum, ownership, and termination protocol.
blackboard           Versioned/provenance-tagged shared artifacts; ACLs and conflict policy are essential.
debate               Bounded counterarguments; only useful with diverse evidence and a verifier.
generator-critic     Proposal challenged against a rubric; critic cannot authorize action.
sequential           Validated deterministic handoffs; predictable but serially 

## 4. Build a scoped blackboard and test conflict resolution

Specialists publish typed artifacts rather than chat: owner, claim, evidence ID, tenant, and calibrated confidence. The board validates scope and records an attributable trail. The critic refuses to convert disagreement into a majority conclusion; it escalates missing, low-confidence, or conflicting evidence. In production, include timestamps, artifact versions, ACLs, correlation IDs, retention, and correction/retraction semantics.

In [3]:
board = Board()
for role, evidence in [('observability', 'metrics-42'), ('deployment', 'deploy-842'), ('customer-impact', 'sla-eu')]:
    board.publish(Artifact(role, 'rollback deploy-842', evidence, 'northstar-eu', .88))
print(critic(board), board.trace)
assert critic(board) == 'proposal:source-supported'

# Deliberate failure: disagreement requests escalation, not endless debate.
conflict = Board()
for role, claim, evidence in [('observability', 'rollback deploy-842', 'metrics-42'), ('deployment', 'hold rollback', 'deploy-842'), ('customer-impact', 'rollback deploy-842', 'sla-eu')]:
    conflict.publish(Artifact(role, claim, evidence, 'northstar-eu', .9))
assert critic(conflict) == 'escalate:conflicting-evidence'

try:
    board.publish(Artifact('observability', 'bad', 'x', 'other-tenant', .8))
except ValueError as error:
    print('blocked:', error)

proposal:source-supported ['publish:observability:metrics-42', 'publish:deployment:deploy-842', 'publish:customer-impact:sla-eu']
blocked: cross-tenant artifact


## 5. Delegation, handoffs, negotiation, consensus, and dynamic teams

**Delegation** retains coordinator accountability while assigning a bounded artifact. **Handoff** transfers conversational control with minimized state and a return condition. **Negotiation** should allocate constrained resources—not decide facts by rhetoric. **Consensus/voting** needs independently comparable candidates, a quorum, abstention, and a conflict path; it cannot erase missing evidence. **Dynamic formation** discovers only approved agents, then chooses the smallest eligible set based on capability, tool/data permissions, tenant/residency, cost, latency, and conflicts of interest.

Every participant needs a task contract: objective, input schema, allowed tools/data, expected output, owner, tenant, correlation ID, deadline, budget, stop condition, and escalation target.

In [4]:
task = 'Investigate a cross-domain EU checkout conversion decline.'
team = allocate(task)
contract = {
    'role': 'deployment', 'allowed_tools': ['read_deployment_history'],
    'expected_artifact': 'source-backed release assessment',
    'tenant': 'northstar-eu', 'budget_calls': 2, 'stop': 'publish or escalate',
}
print(team, contract)
assert set(team) == {'observability', 'deployment', 'customer-impact'}
assert contract['budget_calls'] == 2

('observability', 'deployment', 'customer-impact') {'role': 'deployment', 'allowed_tools': ['read_deployment_history'], 'expected_artifact': 'source-backed release assessment', 'tenant': 'northstar-eu', 'budget_calls': 2, 'stop': 'publish or escalate'}


## 6. Decide with evaluation, not topology preference

Evaluate the exact same task set and policy constraints. Measure source-supported success, task correctness, forbidden actions, p95 latency, total tokens/tool calls, cost per safe success, route/handoff loss, conflict and escalation rate, and recovery behavior. A team wins only when its measured benefit exceeds coordination overhead at your service objective.

The synthetic data below illustrates a plausible trade-off: the team improves cross-domain evidence coverage but costs more. For a simple task, the single baseline still wins.

In [5]:
results = {
    'single': {'supported_success': .78, 'cost': .018, 'p95_s': 5.5, 'policy_failures': 0},
    'team': {'supported_success': .91, 'cost': .034, 'p95_s': 4.6, 'policy_failures': 0},
}
for design, values in results.items():
    values['cost_per_safe_success'] = round(values['cost'] / values['supported_success'], 3)
    print(design, values)
assert results['team']['supported_success'] > results['single']['supported_success']
assert results['team']['cost'] > results['single']['cost']

single {'supported_success': 0.78, 'cost': 0.018, 'p95_s': 5.5, 'policy_failures': 0, 'cost_per_safe_success': 0.023}
team {'supported_success': 0.91, 'cost': 0.034, 'p95_s': 4.6, 'policy_failures': 0, 'cost_per_safe_success': 0.037}


## Production checklist and exercises

- Enforce identity, tenant scope, least privilege, typed schemas, provenance, versioning, idempotency, budgets, timeouts, cancellation, and audit at every message and artifact boundary.
- Cap agent count, turn/message count, delegation depth, concurrency, spending, and replan/debate rounds; define terminal escalation.
- Treat agent messages, discovered capabilities, and shared artifacts as untrusted data until validated.
- Maintain single-agent, deterministic-workflow, and team baselines; do not assume benchmark or demo success transfers to production.

**Exercises:** add a stale evidence policy, a missing specialist timeout, an abstaining weighted vote, a peer-to-peer message TTL, a manager/subagent hierarchy, and an evaluation case where the team must be removed.

References: [LangGraph multi-agent patterns](https://docs.langchain.com/oss/python/langchain/multi-agent/index), [AutoGen teams](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/teams.html), [Multi-Agent Collaboration Mechanisms survey](https://arxiv.org/abs/2501.06322), [AI Agent Systems survey](https://arxiv.org/abs/2601.01743).